## This is a testing Resume.ai notebook

In [23]:
# imports

import os
import torch
from dotenv import load_dotenv
from IPython.display import Markdown

load_dotenv()  # looks for a .env file in the current directory and its parents

True

In [24]:
from huggingface_hub import login

hf_token = os.getenv("HR_TOKEN")
login(hf_token, add_to_git_credential=True)

`add_to_git_credential=True` is only supported when a token is passed directly. It is ignored by the browser-based login.


In [25]:
audio_path = os.path.abspath("../data/LBCA.mp3")
print(audio_path)
audio_filename = open(audio_path, "rb")

/Users/rodrigoec/cc/projetos/resume.ai/data/LBCA.mp3


# Models for transcribing
## pierreguillou/whisper-medium-portuguese

In [26]:
import mlx_whisper

def transcribe(model_id, file_path):
    result = mlx_whisper.transcribe(file_path, path_or_hf_repo=model_id, language="pt")
    return result["text"]

In [28]:
mac_whisper = transcribe("mlx-community/whisper-large-v3-turbo", audio_path)

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 459.50it/s]


In [36]:
Markdown(f"{mac_whisper[:300]}...")

 Eu entrei na LBCA no início de abril de 2026 agora e a minha ideia aqui é meio que contar as experiências que eu tive para depois a gente fazer o resumo dessas informações com o IA. Então quando eu entrei na LBCA eu entrei bem como perfil de intermediação DEV com cientista de dados, então eles sabi...

In [ ]:
mac_whisper_smaller = transcribe("mlx-community/whisper-turbo", audio_path)


In [38]:
Markdown(f"{mac_whisper_smaller[:300]}...")


 Eu entrei na LBCA no início de abril de 2026 agora e a minha ideia aqui é meio que contar as experiências que eu tive para depois a gente fazer o resumo dessas informações com o IA. Então quando eu entrei na LBCA eu entrei bem como perfil de intermediação DEV com cientista de dados, então eles sabi...

## Creating structured topics

In [ ]:
%export os.getenv("ANTHROPIC_API_KEY")

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Topic(BaseModel):
    """Structured topics that describes an experience of a job position using STAR method."""

    company: str = Field(
        description="Company where I worked in on this Topic",
        examples=["LBCA", "ePol", "NuFuturo"],
    )
    situation: str = Field(
        description="The context. What was going on, and what was the problem or need?",
        examples=[
            "Every new client project required rebuilding the document intake logic from scratch."
        ],
    )
    task: int = Field(
        description="your responsibility in that situation. What were you expected to achieve?",
        examples=[
            "Build a reusable service to extract structured data from any document."
        ],
    )
    action: list[str] = Field(
        description="what you specifically did. This is usually the longest part, focused on your decisions, tools, and approach.",
        examples=[
            "Built an OCR + LLM extraction pipeline with AWS Textract and Bedrock, integrated via Kafka."
        ],
    )
    result: str = Field(
        description="the outcome, ideally measurable. What changed because of your work?",
        examples=[
            "The service is now reused across multiple systems, eliminating rework on new projects."
        ],
    )


class JobExperiences(BaseModel):
    """List of job experiences topics"""
    topics: list[Topic] = Field(description="List of STAR topics for the current job")


llm = init_chat_model("anthropic:claude-sonnet-5-5", model_provider="anthropic", api_key=os.getenv("ANTHROPIC_API_KEY"))
structured_llm = llm.with_structured_output(JobExperiences)

result = structured_llm.invoke(
    "I watched Central do Brasil yesterday. Beautiful acting and a touching story, "
    "though a bit slow in the middle. I'd give it 9/10."
)

print(result)  # MovieReview(title='Central do Brasil', rating=9, ...)
print(result.rating)  # 9

TypeError: Anthropic authentication failed: no API key or authorization credentials were provided. Set the ANTHROPIC_API_KEY environment variable, pass api_key=... to ChatAnthropic, or provide credentials via default_headers={"Authorization": ...}. If you are routing through the LangSmith gateway, set LANGSMITH_GATEWAY and LANGSMITH_GATEWAY_API_KEY.